# Lab 03 · Chain the steps of a decision

**Prepared by Mustafa Assaf** · Lecture 2: Chain, call tools, act · 15–20 minute lab

## Break one decision into checked steps

**So far:** Labs 00–02 answered the question with one prompt each: first unguided, then guided by the policy, then grounded by retrieval.

**New:** *prompt chaining*. The decision becomes a fixed sequence of small prompts: **requirements → options → trade-offs → decision record**. Each step's reply is the input of the next step. Between two steps, a *gate* (plain Python) checks the reply and stops the chain before a bad result spreads.

The case is now harder than the course rule. The Payments platform's squads want to release on their own schedules, but only one engineer runs operations. Lectures 2 and 3 move from one prompt to an *architecture council* for this case, one pattern at a time:

- **Lab 03:** one architect's decision, as a chain of checked steps.
- **Lab 04:** tools: functions that read live data or change it.
- **Lab 05:** an agent that chooses its own next step.
- **Lab 06:** three architects on three model families propose alone, challenge each other, and a chair finds the majority.
- **Lab 07:** a human architect reviews the proposal, and a memory keeps the lesson for later cases.
- **Lab 08:** an experiment tests whether the council beats a single model.

**At a glance**

![Lab 03 at a glance: Brief, then Requirements (3 labels), then gate, then Options (2 or 3), then gate, then Trade-offs (+ / − table), then gate, then Decision record (Context · Decision · Consequence), then gate.](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/03.svg)

## The idea in context

A single prompt that asks for a whole decision hides its reasoning. When the answer is wrong, you cannot tell which part failed. Software engineers handle large problems by decomposition: split the work into steps with clear inputs and outputs. Prompt chaining does the same with an LLM: each prompt does one small job, and its output becomes the input of the next prompt.

A chain is a workflow: our code fixes the steps, and the model only fills them in. That makes it predictable, cheap to debug, and easy to test, because a gate can check every intermediate result. Anthropic's guide "Building effective agents" recommends starting with simple workflows like this and adding autonomy only when it is needed. Labs 04 and 05 add that autonomy.

| Key term | Meaning |
| --- | --- |
| **Workflow** | A sequence of steps fixed by code; the model does not choose what happens next. |
| **Decomposition** | Splitting a large task into smaller steps with clear inputs and outputs. |
| **Gate** | A plain-code check between two steps that stops a bad result from spreading. |
| **Decision record (ADR)** | A short note of an architecture decision: its context, the decision, and its consequence. |

## Pattern · Single-Path Planning

*Single-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide. Related: Passive Goal Creator.*

| | |
| --- | --- |
| **Context** | Complex goals need several dependent steps, where each result informs the next. |
| **Problem** | One monolithic response cannot reliably manage dependencies and accumulated intermediate results. |
| **Solution** | Make one ordered plan of a few steps, execute each step with the accumulated context, then synthesize the outputs. |
| **Use when** | The work is naturally sequential; steps depend on earlier outputs; one reasonable path is enough. |
| **Trade-off** | Easy to follow, but an early planning mistake propagates through the whole run. |
| **Flow** | Plan → Step 1 → Step N → Synthesize |
| **Quality attributes** | Benefits: Predictability, Performance, Observability, Maintainability. Liabilities: Reliability, Latency, Usability, Adaptability. |
| **In this lab** | Our code fixes the four-step plan (requirements → options → trade-offs → decision record); a fixed plan like this is often called *prompt chaining*. In the guide's version, the model writes the plan. The gates address the pattern's main liability: they stop an early mistake before it propagates. |

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Practice answers are scripted. For a live model, choose `PROVIDER` (`"freeinference"`, `"gemini"`, or `"deepseek"`), save its key in Colab Secrets (`FREEINFERENCE_API_KEY`, `GEMINI_API_KEY`, or `DEEPSEEK_API_KEY`), and change `MODE` to `"live"`. `ask` stops after 12 calls; run this cell again to reset the count.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit']):
    %pip install -q "agentic-ai-coursekit @ https://github.com/mkassaf/agentic-ai-course/archive/9c78f2181b0f793c60b2b081c79ae74276026e7d.zip"
from coursekit import make_ask
MODE = "practice"
PROVIDER = "freeinference"  # or "gemini" or "deepseek"
ask = make_ask(mode=MODE, provider=PROVIDER)

## Step 1 · The brief, the question, and a link

`run_step` is one link of the chain. It asks the model, prints the reply, and hands it to a *gate*. A gate returns `None` when the reply is fine, or a message that says what is wrong. A failing gate raises an error, so no later step sees the bad reply. A reply that is not JSON makes `json.loads` raise an error inside the gate, which stops the chain too.

`three_items` is the first gate. The last line tries it on a list of two items, without calling the model.

In [ ]:
import json

brief = ("Payments platform: 12 developers in 3 squads; each squad wants to release on its own schedule. "
         "Payments peak at 400 per second, and one engineer runs operations.")
question = "How should the Payments platform be split into deployable units?"

def run_step(name, prompt, example, gate):
    """One link of the chain: ask, show the reply, and let the gate check it before the next link uses it."""
    reply = ask(prompt, example=example)
    print(f"[{name}] {reply}")
    problem = gate(reply)
    if problem:
        raise ValueError(f"The gate after '{name}' stopped the chain: {problem}")
    return reply

def three_items(reply):
    items = json.loads(reply)
    return None if isinstance(items, list) and len(items) == 3 else f"expected a list of 3 items, got {items}"

print(three_items('["independent releases", "low operations effort"]'))  # a gate explains what is wrong

## Step 2 · Requirements, then options

The first link names the three quality requirements that matter most for the question. The second link uses them to name candidate architectures, and its gate accepts two or three *different* options. Each reply is parsed with `json.loads` and becomes the input of the next link.

In [ ]:
prompt = (f"Question: {question}\nBrief: {brief}\nName the 3 quality requirements that matter most "
          "for this question, as short labels. Reply with a JSON list only, no code fences.")
example = '["independent releases", "400 payments per second", "low operations effort"]'
requirements = json.loads(run_step("requirements", prompt, example, three_items))

def two_or_three(reply):
    items = json.loads(reply)
    return None if 2 <= len(set(items)) == len(items) <= 3 else f"expected 2 or 3 different options, got {items}"

prompt = (f"Question: {question}\nBrief: {brief}\nRequirements: {requirements}\nName 2 or 3 candidate "
          "architectures. Reply with a JSON list of short labels only, no code fences.")
options = json.loads(run_step("options", prompt, '["modular monolith", "microservices"]', two_or_three))

## Step 3 · Weigh every option against every requirement

The third link sees only the options and the requirements, not the brief: each link gets just what its step needs. It fills a small table: for each option and each requirement, `"+"` if the option helps and `"-"` if it hurts. The gate uses the replies of the earlier links: it checks that every option has a mark for every requirement, and names any cell that is missing.

In [ ]:
def every_cell(reply):
    table = json.loads(reply)
    missing = [f"{o} / {r}" for o in options for r in requirements if table.get(o, {}).get(r) not in ("+", "-")]
    return f"missing or invalid marks: {missing}" if missing else None

prompt = (f"Options: {options}\nRequirements: {requirements}\nFor each option, mark each requirement "
          '"+" (helps) or "-" (hurts). Reply with JSON only, no code fences, in this shape: '
          '{"<option>": {"<requirement>": "+" or "-"}}')
example = json.dumps({
    "modular monolith": {"independent releases": "-", "400 payments per second": "+", "low operations effort": "+"},
    "microservices": {"independent releases": "+", "400 payments per second": "+", "low operations effort": "-"}})
tradeoffs = json.loads(run_step("trade-offs", prompt, example, every_cell))
for option, marks in tradeoffs.items():
    print(f"   {option:<17}", "   ".join(f"{mark} {requirement}" for requirement, mark in marks.items()))

## Step 4 · Write the decision record

The last link writes a three-line decision record (Context, Decision, and Consequence), using the brief and the trade-off table. Its gate checks the three labels and that the decision is one of the options from Step 2.

In [ ]:
def record_gate(reply):
    fields = dict(line.split(": ", 1) for line in reply.strip().splitlines() if ": " in line)
    if list(fields) != ["Context", "Decision", "Consequence"]:
        return f"expected Context, Decision, Consequence; got {list(fields)}"
    labels = [option.lower() for option in options]
    return None if fields["Decision"].rstrip(".").lower() in labels else f"the Decision must be one of {options}"

prompt = (f"Brief: {brief}\nTrade-offs: {json.dumps(tradeoffs)}\nWrite exactly three lines, starting with "
          '"Context: ", "Decision: " (one option label only), and "Consequence: " (the main cost of the decision).')
example = ("Context: 12 developers in 3 squads want to release on their own schedules.\nDecision: microservices\n"
           "Consequence: Squads release independently, but one operations engineer must run every service.")
record = run_step("decision record", prompt, example, record_gate)

## What you should see

Step 1 prints the gate's message for the two-item list. Step 2 prints three requirements and two options. Step 3 prints the reply and then the same marks as a table: the modular monolith hurts independent releases, and microservices hurt low operations effort. Step 4 prints a decision record for `microservices`, whose Consequence names the cost for the one operations engineer.

The chain made four model calls, and every gate passed. Notice what the gates check: the *shape* of each reply, not whether the decision is wise. The chain saw the operations cost and still chose microservices. One chain gives one architect's view. Lab 06 lets other architects challenge it.

## Try it

1. In Step 2, make the scripted requirements a list of two items. The first gate stops the chain, and the options are never requested.
2. Delete `"low operations effort": "-"` from the microservices row in Step 3. The gate names the missing cell.
3. Change the scripted Decision in Step 4 to `serverless`. Why does the gate reject it, although serverless might be a reasonable design?
4. In live mode, ask for the decision record directly from the brief, in one prompt. Does that record mention the operations cost? Which steps could you no longer check?

## Takeaway

Chaining splits one large request into small steps that you can read and check. Each link sees only what it needs, and a gate stops a bad reply before the next link builds on it. A chain still gives one point of view. Lab 04 gives the model tools, Lab 05 lets it choose its own steps, and Lab 06 adds architects who disagree. Self-study Lab 11 packages a procedure like this decision record as a reusable *skill*.

## Read further

- [Building effective agents: prompt chaining](https://www.anthropic.com/research/building-effective-agents)